In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MHA(nn.Module):
    def __init__(self, d_model, n_heads, causal=True):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_head, self.d_head = n_heads, d_model // n_heads
        self.causal = causal

        self.Wqkv = nn.Linear(d_model, 3 * d_model)
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, d_model = x.shape

        qkv = self.Wqkv(x)  # (B, T, 3 * d_model)
        q, k, v = qkv.chunk(3, dim=-1) # (B, T, d_model)

        q = torch.reshape(q, (B, T, self.n_heads, self.d_head)).transpose(1, 2) # (B, h, T, d_h)
        k = torch.reshape(k, (B, T, self.n_heads, self.d_head)).transpose(1, 2) # (B, h, T, d_h)
        v = torch.reshape(v, (B, T, self.n_heads, self.d_head)).transpose(1, 2) # (B, h, T, d_h)

        attn_score = q @ k.transpose(-2, -1) # (B, h, T, d_h) @ (B, h, d_h, T) -> (B, h, T, T)
        attn_logit = attn_score / math.sqrt(self.d_head) # (B, h, T, T)
        if self.causal:
            mask = torch.ones(T, T, dtype=torch.bool, device=x.device).triu(1)
            attn_logit = attn_logit.masked_fill(mask, float("-inf"))
        attn_logit = attn_logit - attn_logit.amax(dim=-1, keepdim=True) # (B, h, T, T)
        exp_logit = torch.exp(attn_logit) # (B, h, T, T)
        attn_weight = exp_logit / exp_logit.sum(dim=-1, keepdim=True) # (B, h, T, T)
        attn_output = attn_weight @ v # (B, h, T, T) @ (B, h, T, d_h) -> (B, h, T, d_h)

        attn_output = attn_output.transpose(1, 2).contiguous() # (B, T, h, d_h)

        o = torch.reshape(attn_output, (B, T, d_model))
        return self.Wo(o)


In [3]:
x = torch.rand((2, 6, 32))
model = MHA(32, 4, True)
P = 3

with torch.no_grad():
    full   = model(x)
    prefix = model(x[:, :P, :])

diff = (full[:, :P, :] - prefix).abs().max().item()
print(f"max abs diff: {diff:.3e}", torch.allclose(full[:, :P, :], prefix, atol=1e-6))

max abs diff: 0.000e+00 True


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class GQA(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads, causal=True, rope_base=10000.0):
        super().__init__()
        assert d_model % n_q_heads == 0
        assert n_q_heads % n_kv_heads == 0
        self.n_q_heads = n_q_heads
        self.n_kv_heads = n_kv_heads
        self.d_head = d_model // n_q_heads
        self.group_size = n_q_heads // n_kv_heads
        self.causal = causal

        self.Wq = nn.Linear(d_model, n_q_heads * self.d_head)
        self.Wkv = nn.Linear(d_model, 2 * n_kv_heads * self.d_head)
        self.Wo = nn.Linear(d_model, d_model)

        inv_freq = rope_base ** (-torch.arange(0, self.d_head, 2, dtype=torch.float32) / self.d_head)
        self.register_buffer("inv_freq", inv_freq)

    def _apply_rope(self, x, cos, sin):
        # x: [B, h, T, d_h]
        even = x[..., 0::2] # [B, h, T, d_h / 2]
        odd = x[..., 1::2] # [B, h, T, d_h / 2]

        rotated_even = even * cos - odd * sin
        rotated_odd = even * sin + odd * cos

        return torch.stack((rotated_even, rotated_odd), dim=-1).flatten(-2)

    def forward(self, x, positions):
        B, T, _ = x.shape

        q = self.Wq(x)
        k, v = self.Wkv(x).chunk(2, dim=-1)

        q = torch.reshape(q, (B, T, self.n_q_heads, self.d_head)).transpose(1, 2)
        k = torch.reshape(k, (B, T, self.n_kv_heads, self.d_head)).transpose(1, 2)
        v = torch.reshape(v, (B, T, self.n_kv_heads, self.d_head)).transpose(1, 2)

        angles = (positions.to(self.inv_freq.dtype)[:, None] * self.inv_freq[None, :]) # [T, d_head / 2]

        cos = angles.cos()[None, None, :, :] # [1, 1, T, d_head / 2]
        sin = angles.sin()[None, None, :, :] # [1, 1, T, d_head / 2]

        q = self._apply_rope(q, cos, sin)
        k = self._apply_rope(k, cos, sin)

        k_for_attn = k.repeat_interleave(self.group_size, dim=1)
        v_for_attn = v.repeat_interleave(self.group_size, dim=1)

        attn_score = q @ k_for_attn.transpose(-2, -1)
        attn_logits = attn_score / math.sqrt(self.d_head)
        if self.causal:
            query_positions = positions[:, None]
            key_positions = positions[None, :]
            forbidden = key_positions > query_positions
            attn_logits = attn_logits.masked_fill(forbidden, float("-inf"))
        attn_logits = attn_logits - attn_logits.amax(dim=-1, keepdim=True)
        exp_logits = torch.exp(attn_logits)
        attn_weights = exp_logits / exp_logits.sum(dim=-1, keepdim=True)
        attn_out = attn_weights @ v_for_attn

        attn_out = attn_out.transpose(1, 2).contiguous()
        o = torch.reshape(attn_out, (B, T, self.n_q_heads * self.d_head))

        return self.Wo(o)

In [7]:
attn = GQA(d_model=256, n_q_heads=8, n_kv_heads=2)
x = torch.randn(2, 6, 256)
positions = torch.arange(6, device=x.device)

out = attn(x, positions)  # [2, 6, 256]

In [8]:
torch.manual_seed(0)

model = GQA(d_model=256, n_q_heads=8, n_kv_heads=2)
vectors = torch.randn(2, 8, 6, model.d_head)


def rotate(x, positions):
    angles = (
        positions.to(model.inv_freq.dtype)[:, None]
        * model.inv_freq[None, :]
    )
    cos = angles.cos()[None, None, :, :]
    sin = angles.sin()[None, None, :, :]
    return model._apply_rope(x, cos, sin)


# 1. Position zero means zero rotation.
zero_rotated = rotate(vectors, torch.zeros(6, dtype=torch.long))
torch.testing.assert_close(zero_rotated, vectors)

# 2. Rotation preserves each head vector's squared length.
rotated = rotate(vectors, torch.arange(6))
torch.testing.assert_close(
    rotated.square().sum(dim=-1),
    vectors.square().sum(dim=-1),
    atol=1e-5,
    rtol=1e-5,
)

# 3. Splitting execution preserves results when positions are preserved.
chunked = torch.cat(
    [
        rotate(vectors[:, :, :3, :], torch.arange(3)),
        rotate(vectors[:, :, 3:, :], torch.arange(3, 6)),
    ],
    dim=2,
)
torch.testing.assert_close(chunked, rotated)

print("RoPE property checks passed")

RoPE property checks passed


In [1]:
import math
import torch
import torch.nn as nn

class KVCache:
    def __init__(self, batch_size, n_kv_heads, capacity, d_head, *, device, dtype):
        shape = (batch_size, n_kv_heads, capacity, d_head)
        self.k = torch.empty(shape, device=device, dtype=dtype)
        self.v = torch.empty(shape, device=device, dtype=dtype)
        self.capacity = capacity
        self.length = 0

    def append(self, k_new, v_new):
        # incoming tensor shapes: [B, Hkv, T, d]
        if k_new.ndim != 4 or v_new.shape != k_new.shape:
            raise ValueError("K/V must have matching [B, Hkv, T, d] shapes")

        B, H, T, d = k_new.shape

        start = self.length
        end = start + T

        self.k[:, :, start:end, :].copy_(k_new)
        self.v[:, :, start:end, :].copy_(v_new)
        self.length = end

        return self.k[:, :, :end, :], self.v[:, :, :end, :]

    def reset(self):
        self.length = 0

class GQA(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads, causal=True, rope_base=10000.0):
        super().__init__()
        self.n_q_heads = n_q_heads
        self.n_kv_heads = n_kv_heads
        self.d_head = d_model // n_q_heads
        self.group_size = n_q_heads // n_kv_heads
        self.causal = causal

        self.Wq = nn.Linear(d_model, n_q_heads * self.d_head)
        self.Wkv = nn.Linear(d_model, 2 * n_kv_heads * self.d_head)
        self.Wo = nn.Linear(d_model, d_model)

        inv_freq = rope_base ** (
            -torch.arange(0, self.d_head, 2, dtype=torch.float32)
            / self.d_head
        )
        self.register_buffer("inv_freq", inv_freq)

    def _apply_rope(self, x, cos, sin):
        even = x[..., 0::2]
        odd = x[..., 1::2]

        rotated_even = even * cos - odd * sin
        rotated_odd = even * sin + odd * cos

        return torch.stack(
            (rotated_even, rotated_odd), dim=-1
        ).flatten(-2)

    def forward(self, x, positions, cache=None):
        B, T, _ = x.shape

        if cache is not None:
            expected = torch.arange(
                cache.length,
                cache.length + T,
                device=x.device,
                dtype=positions.dtype,
            )

        q = self.Wq(x)
        k, v = self.Wkv(x).chunk(2, dim=-1)

        q = q.reshape(
            B, T, self.n_q_heads, self.d_head
        ).transpose(1, 2)  # [B, Hq, T, d]

        k = k.reshape(
            B, T, self.n_kv_heads, self.d_head
        ).transpose(1, 2)  # [B, Hkv, T, d]

        v = v.reshape(
            B, T, self.n_kv_heads, self.d_head
        ).transpose(1, 2)  # [B, Hkv, T, d]

        angles = (
            positions.to(self.inv_freq.dtype)[:, None]
            * self.inv_freq[None, :]
        )
        cos = angles.cos()[None, None, :, :]
        sin = angles.sin()[None, None, :, :]

        q = self._apply_rope(q, cos, sin)
        k = self._apply_rope(k, cos, sin)

        if cache is not None:
            # Store NEW rotated K and NEW V; retrieve ALL populated K/V.
            k, v = cache.append(k, v)
            key_positions = torch.arange(
                cache.length, device=x.device, dtype=positions.dtype
            )
        else:
            key_positions = positions

        # k/v: [B, Hkv, S, d], where S = P + T for cached calls.
        k_for_attn = k.repeat_interleave(self.group_size, dim=1)
        v_for_attn = v.repeat_interleave(self.group_size, dim=1)

        logits = (
            q @ k_for_attn.transpose(-2, -1)
        ) / math.sqrt(self.d_head)  # [B, Hq, T, S]

        if self.causal:
            forbidden = key_positions[None, :] > positions[:, None]
            logits = logits.masked_fill(forbidden, float("-inf"))

        weights = torch.softmax(logits, dim=-1)
        out = weights @ v_for_attn  # [B, Hq, T, d]

        out = (
            out.transpose(1, 2)
            .contiguous()
            .view(B, T, self.n_q_heads * self.d_head)
        )
        return self.Wo(out)
        

In [2]:
torch.manual_seed(0)

model = GQA(d_model=256, n_q_heads=8, n_kv_heads=2).eval()
x = torch.randn(2, 6, 256)
positions = torch.arange(6, device=x.device)

cache = KVCache(
    batch_size=2,
    n_kv_heads=model.n_kv_heads,
    capacity=16,
    d_head=model.d_head,
    device=x.device,
    dtype=x.dtype,
)

with torch.inference_mode():
    reference = model(x, positions)

    for chunks in ([6], [1, 1, 1, 1, 1, 1], [3, 1, 2]):
        cache.reset()
        outputs = []
        start = 0

        for size in chunks:
            end = start + size
            outputs.append(
                model(
                    x[:, start:end, :],
                    positions[start:end],
                    cache=cache,
                )
            )
            start = end
            assert cache.length == end

        cached = torch.cat(outputs, dim=1)

        torch.testing.assert_close(
            cached, reference, atol=1e-5, rtol=1e-4
        )
        error = (cached - reference).abs().max().item()
        print(f"chunks={chunks}: max abs diff={error:.3e}")
        

chunks=[6]: max abs diff=0.000e+00
chunks=[1, 1, 1, 1, 1, 1]: max abs diff=5.960e-07
chunks=[3, 1, 2]: max abs diff=2.980e-07
